# 🚀 Coffee Chain Demand Forecasting - Mamba SSM Hybrid (v2: Deep Context Edition)
This notebook utilizes **Mamba (State Space Model)** to extract deep temporal patterns from the raw 84-day historical sequence, combined with an MLP for 45+ tabular covariates (promos, events, annual lags, and store capacity).

> ⚠️ **REQUIREMENT:** This notebook MUST be run with a GPU (e.g., Kaggle T4x2 or P100). `mamba-ssm` requires CUDA to compile and run its fast kernels.


In [1]:
# Install dependencies for Mamba SSM (Fix for Kaggle Wheel Building)
import subprocess
import sys

try:
    import mamba_ssm
    print("mamba-ssm is already installed!")
except ImportError:
    print("Installing packaging and ninja to support building causal-conv1d/mamba-ssm...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "packaging", "ninja", "--quiet"])
    print("Installing causal-conv1d and mamba-ssm...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "causal-conv1d>=1.4.0", "mamba-ssm", "--no-build-isolation", "--quiet"])
    print("Installation complete!")

import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from mamba_ssm import Mamba

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Installing packaging and ninja to support building causal-conv1d/mamba-ssm...
Installing causal-conv1d and mamba-ssm...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.4/216.4 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 MB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.8/260.8 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 MB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.3/29.3 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.8/897.8 kB 51.3 MB/s eta 0:00:00
Installation complete!
Using device: cuda


## 1. Data Setup & Loading


In [2]:
DATA_DIR_CANDIDATES = [
    Path(os.environ.get("COFFEE_DATA_DIR", "")),
    Path("/content/super-ai-engineer-season-6-coffee-chain-hackathon"),
    Path("/kaggle/input/super-ai-engineer-season-6-coffee-chain-hackathon"),
    Path("/kaggle/input/competitions/super-ai-engineer-season-6-coffee-chain-hackathon"),
    Path(r"c:\Users\CPE KMUTT\Documents\GitHub\superai_engineer_ss6\Level 2\Hackathon 5_Demand Forecasting Coffee Chain Hackathon")
]
DATA_DIR = next((p for p in DATA_DIR_CANDIDATES if (p / "train").exists()), DATA_DIR_CANDIDATES[-1])

OUTPUT_PATH = Path("submission_mamba_hybrid.csv")

TRAIN_START = pd.Timestamp("2023-01-01")
TRAIN_END = pd.Timestamp("2024-10-31")
VALID_START = pd.Timestamp("2023-12-01")
VALID_END = pd.Timestamp("2023-12-31")

HORIZON_TO_DAYS = {"1d": 1, "7d": 7, "1m": 30}
SEQ_LEN = 84 # Look back 12 weeks
BATCH_SIZE = 128
EPOCHS = 24
LR = 1e-3

def read_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    return df

train_dir = DATA_DIR / "train"
test_dir = DATA_DIR / "test"

txn = read_csv(train_dir / "TRANSACTION.csv")
order = read_csv(train_dir / "ORDER.csv")
inventory = read_csv(train_dir / "INVENTORY.csv")
promo_train = read_csv(train_dir / "PROMOTION.csv")
event_train = read_csv(train_dir / "LOCAL_EVENT.csv")
date_train = read_csv(train_dir / "DATE_DIM.csv")
store_train = read_csv(train_dir / "STORE.csv")
product_train = read_csv(train_dir / "PRODUCT.csv")

promo_test = read_csv(test_dir / "PROMOTION.csv")
event_test = read_csv(test_dir / "LOCAL_EVENT.csv")
date_test = read_csv(test_dir / "DATE_DIM.csv")
store_test = read_csv(test_dir / "STORE.csv")
product_test = read_csv(test_dir / "PRODUCT.csv")

sample = pd.read_csv(DATA_DIR / "sample_submission.csv") if (DATA_DIR / "sample_submission.csv").exists() else pd.read_csv(DATA_DIR / "sample_submission_with_id.csv")

order["date"] = pd.to_datetime(order["date"])
inventory["date"] = pd.to_datetime(inventory["date"])
date_train["date"] = pd.to_datetime(date_train["date"])
date_test["date"] = pd.to_datetime(date_test["date"])

## 2. Advanced Feature Engineering (Deep Context)
Injecting insights like cross-promo cannibalization, store capacity, and event interaction effects.


In [3]:
# Target Grid
target = (
    txn[["order_id", "product_id", "units_sold"]]
    .merge(order[["order_id", "store_id", "date"]], on="order_id", how="left")
    .merge(product_train[["product_id", "category"]], on="product_id", how="left")
    .groupby(["store_id", "category", "date"], as_index=False)["units_sold"].sum()
)
target["store_id"] = target["store_id"].astype(int)
target["date"] = pd.to_datetime(target["date"])
target["units_sold"] = target["units_sold"].astype("float32")

stores = sorted(target["store_id"].unique())
categories = sorted(target["category"].unique())
dates = pd.date_range(TRAIN_START, TRAIN_END, freq="D")
date_to_idx = {d: i for i, d in enumerate(dates)}

grid = pd.MultiIndex.from_product([stores, categories, dates], names=["store_id", "category", "date"]).to_frame(index=False)
full = grid.merge(target, on=["store_id", "category", "date"], how="left")
full["units_sold"] = full["units_sold"].fillna(0).astype("float32")

# Stockout
inv = inventory.merge(product_train[["product_id", "category"]], on="product_id", how="left")
stockout_cat = inv.groupby(["store_id", "category", "date"], as_index=False).agg(stockout_rate=("is_stockout", "mean"))
full = full.merge(stockout_cat, on=["store_id", "category", "date"], how="left")
full["stockout_rate"] = full["stockout_rate"].fillna(0).astype("float32")

# Store Features
store_df = pd.concat([store_train, store_test], ignore_index=True).drop_duplicates("store_id")
store_df["open_hour"] = pd.to_numeric(store_df["open_time"].astype(str).str.split(":").str[0], errors="coerce").fillna(7)
store_df["close_hour"] = pd.to_numeric(store_df["close_time"].astype(str).str.split(":").str[0], errors="coerce").fillna(21)
store_df["operating_hours"] = store_df["close_hour"] - store_df["open_hour"]
store_df["capacity_per_staff"] = store_df["seating_capacity"] / store_df["staff_count"].replace(0, np.nan)
store_df["has_drive_through"] = store_df["has_drive_through"].astype(str).str.lower().isin(["true", "1", "y", "yes"]).astype(int)

neighborhood_dummies = pd.get_dummies(store_df["neighborhood_type"], prefix="nbr").astype(int)
expected_nbrs = ["nbr_airport", "nbr_business", "nbr_hospital", "nbr_mall", "nbr_residential", "nbr_tourist", "nbr_university"]
for col in expected_nbrs:
    if col not in neighborhood_dummies: neighborhood_dummies[col] = 0
store_features = pd.concat([store_df[["store_id", "seating_capacity", "staff_count", "operating_hours", "capacity_per_staff", "has_drive_through"]], neighborhood_dummies], axis=1)

# Date Features
date_all = pd.concat([date_train, date_test], ignore_index=True).drop_duplicates("date")
date_all["date"] = pd.to_datetime(date_all["date"])
date_all["dow"] = date_all["date"].dt.dayofweek
date_all["doy"] = date_all["date"].dt.dayofyear
date_all["month"] = date_all["date"].dt.month
date_all["sin_dow"] = np.sin(2 * np.pi * date_all["dow"] / 7.0)
date_all["cos_dow"] = np.cos(2 * np.pi * date_all["dow"] / 7.0)
date_all["sin_doy"] = np.sin(2 * np.pi * date_all["doy"] / 365.25)
date_all["cos_doy"] = np.cos(2 * np.pi * date_all["doy"] / 365.25)
for col in ["is_weekend", "is_holiday", "is_payday", "is_school_break", "is_rainy_season"]:
    date_all[col] = date_all.get(col, 0).astype(str).str.lower().isin(["true", "1", "y", "yes"]).astype(int)

# Annual Lags
annual_lags = full[["store_id", "category", "date", "units_sold"]].copy()
annual_lags = annual_lags.sort_values(["store_id", "category", "date"])
for _lag in [364, 365, 366]:
    annual_lags[f"lag_{_lag}"] = annual_lags.groupby(["store_id", "category"])["units_sold"].shift(_lag).fillna(0)
annual_lags = annual_lags.drop(columns=["units_sold"])
full = full.merge(annual_lags, on=["store_id", "category", "date"], how="left")
for _lag in [364, 365, 366]: full[f"lag_{_lag}"] = full[f"lag_{_lag}"].fillna(0)

# Promo Features
promo = pd.concat([promo_train, promo_test], ignore_index=True).drop_duplicates()
promo = promo.merge(product_train[["product_id", "category"]], on="product_id", how="left")
promo["start_date"] = pd.to_datetime(promo["start_date"])
promo["end_date"] = pd.to_datetime(promo["end_date"])

text_cols = [c for c in ["campaign", "campaign_type", "campaign_name", "campaign_id"] if c in promo.columns]
if text_cols:
    pt = promo[text_cols[0]].fillna("").astype(str)
    for c in text_cols[1:]: pt = pt + " " + promo[c].fillna("").astype(str)
    promo["promo_text"] = pt.str.lower()
else:
    promo["promo_text"] = ""

promo["is_buy1get1"] = promo["promo_text"].str.contains("1แถม1|buy1get1|b1g1|1 แถม 1").astype(int)
promo["is_discount"] = promo["promo_text"].str.contains("ลดราคา|discount").astype(int)
promo["is_bundle"] = promo["promo_text"].str.contains("ชุดคู่|bundle").astype(int)

promo_rows = []
for _, r in promo.iterrows():
    if pd.isna(r["start_date"]) or pd.isna(r["category"]): continue
    start = max(pd.Timestamp(r["start_date"]), TRAIN_START)
    end = min(pd.Timestamp(r["end_date"]), pd.Timestamp("2024-12-31"))
    if end < start: continue
    store_values = stores if pd.isna(r.get("store_id")) else [int(r["store_id"])]
    for store_id in store_values:
        for d in pd.date_range(start, end, freq="D"):
            promo_rows.append({
                "store_id": store_id, "category": str(r["category"]), "date": d,
                "promo_count": 1, 
                "discount_pct": float(0 if pd.isna(r.get("discount_pct")) else r["discount_pct"]),
                "is_buy1get1": r["is_buy1get1"],
                "is_discount": r["is_discount"],
                "is_bundle": r["is_bundle"]
            })
p_df = pd.DataFrame(promo_rows)
if len(p_df) > 0:
    promo_features = p_df.groupby(["store_id", "category", "date"], as_index=False).agg(
        promo_count=("promo_count", "sum"),
        discount_pct=("discount_pct", "max"),
        is_buy1get1=("is_buy1get1", "max"),
        is_discount=("is_discount", "max"),
        is_bundle=("is_bundle", "max")
    )
    # Cross-promo features
    p_df["is_drink"] = p_df["category"].isin(["Coffee", "Tea", "Juice & Smoothie", "Chocolate & Milk"]).astype(int)
    p_df["is_merch"] = (p_df["category"] == "Merchandise").astype(int)
    store_promo = p_df.groupby(["store_id", "date"], as_index=False).agg(
        drink_promo_active=("is_drink", "max"),
        merch_promo_active=("is_merch", "max")
    )
    promo_features = promo_features.merge(store_promo, on=["store_id", "date"], how="left")
else:
    promo_features = pd.DataFrame(columns=["store_id", "category", "date", "promo_count", "discount_pct", "is_buy1get1", "is_discount", "is_bundle", "drink_promo_active", "merch_promo_active"])

# Event
event = pd.concat([event_train, event_test], ignore_index=True).drop_duplicates()
event["date"] = pd.to_datetime(event["date"])
event_agg = event.groupby(["store_id", "date"]).agg(
    event_count=("event_id", "nunique"),
    event_types=("event_type", lambda x: " ".join(x.dropna().astype(str).str.lower()))
).reset_index()
event_agg["event_food_festival"] = event_agg["event_types"].str.contains("food_festival|food festival").astype(int)
event_agg["event_music_festival"] = event_agg["event_types"].str.contains("music|concert").astype(int)
event_agg["event_cultural"] = event_agg["event_types"].str.contains("cultural|culture").astype(int)
event_agg["event_sports"] = event_agg["event_types"].str.contains("sports|marathon").astype(int)
event_features = event_agg.drop(columns=["event_types"])

# Merge all into FULL
full = full.merge(store_features, on="store_id", how="left")
date_cols = ["date", "sin_dow", "cos_dow", "sin_doy", "cos_doy", "is_weekend", "is_holiday", "is_payday", "is_school_break", "is_rainy_season"]
full = full.merge(date_all[date_cols], on="date", how="left")
full = full.merge(promo_features, on=["store_id", "category", "date"], how="left")
full = full.merge(event_features, on=["store_id", "date"], how="left")
full = full.fillna(0)

# Interaction Features
full["is_coffee"] = (full["category"] == "Coffee").astype(int)
full["is_juice"] = (full["category"] == "Juice & Smoothie").astype(int)
full["rainy_x_coffee"] = full["is_rainy_season"] * full["is_coffee"]
full["rainy_x_juice"] = full["is_rainy_season"] * full["is_juice"]

## 3. Fast Dense Sequence Tensor Preparation
Now with expanded daily sequence channels!


In [4]:
store_cat_to_idx = { (s, c): i for i, (s, c) in enumerate([(s, c) for s in stores for c in categories]) }
num_series = len(store_cat_to_idx)
num_days = len(dates)

# Sequence Features (Expanded to 12 channels)
seq_features = ["units_sold", "stockout_rate", "promo_count", "discount_pct", 
                "is_buy1get1", "is_holiday", "is_payday", "is_rainy_season", 
                "is_school_break", "event_count", "sin_dow", "cos_dow"]
num_seq_features = len(seq_features)

seq_tensor = np.zeros((num_series, num_days, num_seq_features), dtype=np.float32)

full["series_idx"] = full.apply(lambda r: store_cat_to_idx[(r["store_id"], r["category"])], axis=1)
full["time_idx"] = full["date"].map(date_to_idx)

for i, f in enumerate(seq_features):
    seq_tensor[full["series_idx"].values, full["time_idx"].values, i] = full[f].values

print("Sequence Tensor Shape:", seq_tensor.shape)

Sequence Tensor Shape: (140, 670, 12)


## 4. PyTorch Dataset
Fusing the history tensor with 40+ tabular context features.


In [5]:
tabular_cols = [
    "sin_dow", "cos_dow", "sin_doy", "cos_doy", "is_weekend", "is_holiday", 
    "is_payday", "is_rainy_season", "is_school_break",
    "promo_count", "discount_pct", "is_buy1get1", "is_discount", "is_bundle", 
    "drink_promo_active", "merch_promo_active",
    "event_count", "event_food_festival", "event_music_festival", "event_cultural", "event_sports",
    "seating_capacity", "staff_count", "operating_hours", "capacity_per_staff", "has_drive_through",
    "nbr_airport", "nbr_business", "nbr_hospital", "nbr_mall", "nbr_residential", "nbr_tourist", "nbr_university",
    "lag_364", "lag_365", "lag_366",
    "is_coffee", "is_juice", "rainy_x_coffee", "rainy_x_juice"
]
NUM_TAB_FEATURES = len(tabular_cols)

class CoffeeDataset(Dataset):
    def __init__(self, target_df, seq_tensor, date_to_idx, store_cat_to_idx, mode="train"):
        self.target_df = target_df.reset_index(drop=True)
        self.seq_tensor = seq_tensor
        self.date_to_idx = date_to_idx
        self.store_cat_to_idx = store_cat_to_idx
        self.mode = mode
        
    def __len__(self):
        return len(self.target_df)
    
    def __getitem__(self, idx):
        row = self.target_df.iloc[idx]
        sid, cat = row["store_id"], row["category"]
        anchor_date = row["anchor_date"]
        
        series_i = self.store_cat_to_idx[(sid, cat)]
        t_end = self.date_to_idx.get(anchor_date)
        if t_end is None: t_end = self.date_to_idx[TRAIN_END]
            
        t_start = t_end - SEQ_LEN
        
        if t_start >= 0:
            seq_x = self.seq_tensor[series_i, t_start:t_end, :]
        else:
            seq_x = np.zeros((SEQ_LEN, self.seq_tensor.shape[2]), dtype=np.float32)
            valid_len = t_end
            if valid_len > 0:
                seq_x[-valid_len:, :] = self.seq_tensor[series_i, 0:t_end, :]
                
        tab_x = np.array([row.get(c, 0) for c in tabular_cols], dtype=np.float32)
        
        weight = 1.0
        if self.mode == "train":
            rate = row.get("target_stockout_rate", 0)
            weight = 1.0 - (1.0 - 0.30) * min(max(rate, 0), 1)
            
        if self.mode == "test":
            return torch.tensor(seq_x), torch.tensor(tab_x)
        else:
            y = np.float32(row["target_units_sold"])
            return torch.tensor(seq_x), torch.tensor(tab_x), torch.tensor(y), torch.tensor(weight)

## 5. Mamba Hybrid Model Architecture


In [6]:
class MambaHybrid(nn.Module):
    def __init__(self, seq_in_dim, tab_in_dim, d_model=128, d_state=32, d_conv=4, expand=2):
        super().__init__()
        
        self.seq_proj = nn.Linear(seq_in_dim, d_model)
        
        self.mamba = Mamba(
            d_model=d_model,
            d_state=d_state,
            d_conv=d_conv,
            expand=expand,
        )
        
        self.tab_proj = nn.Sequential(
            nn.Linear(tab_in_dim, 64),
            nn.GELU(),
            nn.LayerNorm(64),
            nn.Linear(64, 32)
        )
        
        self.head = nn.Sequential(
            nn.Linear(d_model + 32, 128),
            nn.GELU(),
            nn.LayerNorm(128),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Linear(64, 1)
        )
        
    def forward(self, seq_x, tab_x):
        x = self.seq_proj(seq_x)
        mamba_out = self.mamba(x)
        seq_embed = mamba_out.mean(dim=1)
        tab_embed = self.tab_proj(tab_x)
        
        fused = torch.cat([seq_embed, tab_embed], dim=1)
        out = self.head(fused)
        
        return out.squeeze(-1)

## 6. Build Train/Test DataFrames & Future Join


In [7]:
def parse_submission(sample):
    out = sample.copy()
    parsed = out["id"].str.extract(r"^(\d+)_(.+)_(\d{4}-\d{2}-\d{2})_(1d|7d|1m)$")
    out[["store_id", "category", "forecast_date", "horizon"]] = parsed
    out["store_id"] = out["store_id"].astype(int)
    out["forecast_date"] = pd.to_datetime(out["forecast_date"])
    out["h"] = out["horizon"].map(HORIZON_TO_DAYS).astype(int)
    out["decision_date"] = out["forecast_date"] - pd.to_timedelta(out["h"], unit="D")
    return out

parsed_sample = parse_submission(sample)

# Build Future lookup dataframe
future_features = full[["store_id", "category", "date"] + tabular_cols].copy()
future_features = future_features.rename(columns={"date": "forecast_date"})

def add_future_covariates(df):
    out = df.merge(future_features, on=["store_id", "category", "forecast_date"], how="left")
    return out.fillna(0)

train_frames = []
for horizon, h in HORIZON_TO_DAYS.items():
    base = full[["store_id", "category", "date", "units_sold", "stockout_rate"]].rename(
        columns={"date": "forecast_date", "units_sold": "target_units_sold", "stockout_rate": "target_stockout_rate"}
    )
    base["horizon"] = horizon
    base["h"] = h
    base["decision_date"] = base["forecast_date"] - pd.to_timedelta(h, unit="D")
    base["anchor_date"] = base["decision_date"]
    
    base = base[(base["anchor_date"] >= TRAIN_START) & (base["forecast_date"] <= TRAIN_END)].copy()
    base = add_future_covariates(base)
    train_frames.append(base)

train_all = pd.concat(train_frames, ignore_index=True)

test_all = parsed_sample.copy()
test_all["anchor_date"] = test_all["decision_date"].clip(upper=TRAIN_END)
test_all = add_future_covariates(test_all)

## 7. Training Loop


In [8]:
def train_horizon(horizon):
    print(f"\n{'='*40}\nTraining Mamba for Horizon: {horizon}\n{'='*40}")
    df_h = train_all[train_all["horizon"] == horizon].copy()
    
    is_valid = df_h["forecast_date"].between(VALID_START, VALID_END)
    train_df = df_h[~is_valid]
    valid_df = df_h[is_valid]
    
    train_ds = CoffeeDataset(train_df, seq_tensor, date_to_idx, store_cat_to_idx, mode="train")
    valid_ds = CoffeeDataset(valid_df, seq_tensor, date_to_idx, store_cat_to_idx, mode="valid")
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False)
    
    model = MambaHybrid(seq_in_dim=num_seq_features, tab_in_dim=NUM_TAB_FEATURES).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    criterion = nn.L1Loss(reduction='none')
    
    best_mae = float('inf')
    best_weights = None
    
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0.0
        for seq_x, tab_x, y, weight in train_loader:
            seq_x, tab_x, y, weight = seq_x.to(device), tab_x.to(device), y.to(device), weight.to(device)
            
            optimizer.zero_grad()
            preds = model(seq_x, tab_x)
            loss = (criterion(preds, y) * weight).mean()
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * seq_x.size(0)
            
        train_loss /= len(train_loader.dataset)
        
        model.eval()
        valid_loss = 0.0
        with torch.no_grad():
            for seq_x, tab_x, y, weight in valid_loader:
                seq_x, tab_x, y = seq_x.to(device), tab_x.to(device), y.to(device)
                preds = torch.relu(model(seq_x, tab_x))
                loss = nn.L1Loss()(preds, y)
                valid_loss += loss.item() * seq_x.size(0)
        valid_loss /= len(valid_loader.dataset)
        
        print(f"Epoch {epoch+1:02d} | Train L1: {train_loss:.4f} | Valid MAE: {valid_loss:.4f}")
        
        if valid_loss < best_mae:
            best_mae = valid_loss
            best_weights = model.state_dict().copy()
            
    print(f"Best Valid MAE: {best_mae:.4f}")
    
    print("Retraining on full data...")
    full_ds = CoffeeDataset(df_h, seq_tensor, date_to_idx, store_cat_to_idx, mode="train")
    full_loader = DataLoader(full_ds, batch_size=BATCH_SIZE, shuffle=True)
    
    model.load_state_dict(best_weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR/2)
    
    model.train()
    for epoch in range(2):
        for seq_x, tab_x, y, weight in full_loader:
            seq_x, tab_x, y, weight = seq_x.to(device), tab_x.to(device), y.to(device), weight.to(device)
            optimizer.zero_grad()
            preds = model(seq_x, tab_x)
            loss = (criterion(preds, y) * weight).mean()
            loss.backward()
            optimizer.step()
            
    return model

trained_models = {}
for horizon in ["1d", "7d", "1m"]:
    trained_models[horizon] = train_horizon(horizon)


Training Mamba for Horizon: 1d
Epoch 01 | Train L1: 15.6443 | Valid MAE: 18.5554
Epoch 02 | Train L1: 12.2318 | Valid MAE: 17.9580
Epoch 03 | Train L1: 11.0732 | Valid MAE: 15.7931
Epoch 04 | Train L1: 10.6968 | Valid MAE: 16.1339
Epoch 05 | Train L1: 10.3660 | Valid MAE: 15.7153
Epoch 06 | Train L1: 10.0757 | Valid MAE: 17.4209
Epoch 07 | Train L1: 9.8403 | Valid MAE: 15.1754
Epoch 08 | Train L1: 9.6535 | Valid MAE: 15.8216
Epoch 09 | Train L1: 9.6056 | Valid MAE: 14.8572
Epoch 10 | Train L1: 9.4734 | Valid MAE: 16.0290
Epoch 11 | Train L1: 9.3685 | Valid MAE: 13.5428
Epoch 12 | Train L1: 9.3160 | Valid MAE: 14.7330
Best Valid MAE: 13.5428
Retraining on full data...

Training Mamba for Horizon: 7d
Epoch 01 | Train L1: 15.7705 | Valid MAE: 19.0341
Epoch 02 | Train L1: 11.5932 | Valid MAE: 16.3751
Epoch 03 | Train L1: 10.9341 | Valid MAE: 16.2099
Epoch 04 | Train L1: 10.5734 | Valid MAE: 14.6840
Epoch 05 | Train L1: 10.3497 | Valid MAE: 16.4229
Epoch 06 | Train L1: 10.0096 | Valid MAE:

## 8. Inference & Submission


In [9]:
submission = sample[["id"]].copy()
submission["units_sold_predicted"] = np.nan

for horizon in ["1d", "7d", "1m"]:
    idx = test_all["horizon"] == horizon
    test_df = test_all.loc[idx]
    
    test_ds = CoffeeDataset(test_df, seq_tensor, date_to_idx, store_cat_to_idx, mode="test")
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)
    
    model = trained_models[horizon]
    model.eval()
    
    all_preds = []
    with torch.no_grad():
        for seq_x, tab_x in test_loader:
            seq_x, tab_x = seq_x.to(device), tab_x.to(device)
            preds = torch.relu(model(seq_x, tab_x))
            all_preds.extend(preds.cpu().numpy().tolist())
            
    submission.loc[idx.values, "units_sold_predicted"] = all_preds
    print(f"Horizon {horizon} predictions mean: {np.mean(all_preds):.4f}")

submission.to_csv(OUTPUT_PATH, index=False)
print("\nSaved to", OUTPUT_PATH)
display(submission.head())

Horizon 1d predictions mean: 18.1207
Horizon 7d predictions mean: 21.8098
Horizon 1m predictions mean: 17.7016

Saved to submission_mamba_hybrid.csv


,id,units_sold_predicted
0,1_Bakery_2024-11-01_1d,9.120278
1,1_Bakery_2024-11-01_1m,9.213926
2,1_Bakery_2024-11-01_7d,13.815886
3,1_Bakery_2024-11-02_1d,9.120278
4,1_Bakery_2024-11-02_1m,9.366965
